In [1]:
%run ragguard_resume.py


Exception: File `'ragguard_resume.py'` not found.

In [2]:
%run "C:\Users\yukta ghara\Downloads\ragguard_resume.py"

Saved: C:\Users\yukta ghara\RAGGuard_Project\data\source_info.jsonl (15,117,971 bytes)
Saved: C:\Users\yukta ghara\RAGGuard_Project\data\response.jsonl (21,458,735 bytes)

Loading JSONL datasets ...
Sources shape:   (2965, 5)
Responses shape: (17790, 8)

Required columns: OK
Source IDs: OK - every response has a matching source

RAGGUARD DATASET SUMMARY
Number of source records:   2,965
Number of response records: 17,790

Task type counts:
task_type
Data2txt    1033
QA           989
Summary      943

Model counts:
model
gpt-4-0613             2965
gpt-3.5-turbo-0613     2965
mistral-7B-instruct    2965
llama-2-7b-chat        2965
llama-2-13b-chat       2965
llama-2-70b-chat       2965

Original split counts:
split
train    15090
test      2700

Missing values in source dataset:
source_id      0
task_type      0
source         0
source_info    0
prompt         0

Missing values in response dataset:
id             0
source_id      0
model          0
temperature    0
labels         0
spli

In [3]:
import pandas as pd
import json

# Load the saved dataset
df = pd.read_parquet(
    r"C:\Users\yukta ghara\RAGGuard_Project\processed\ragguard_merged.parquet"
)

# Find the first response containing hallucination annotations
sample = df[df["is_hallucinated"] == True].iloc[0]

print("RESPONSE ID:", sample["id"])
print("SOURCE ID:", sample["source_id"])

print("\nRESPONSE TEXT:")
print(sample["response"][:1000])

print("\nANNOTATION TYPE:", type(sample["labels"]))
print("NUMBER OF ANNOTATIONS:", len(sample["labels"]))

print("\nFIRST THREE ANNOTATIONS:")
for annotation in sample["labels"][:3]:
    print(json.dumps(annotation, indent=4, ensure_ascii=False))

RESPONSE ID: 2
SOURCE ID: 15592

RESPONSE TEXT:
New research conducted by the Anne Frank House has revealed that Anne Frank and her sister Margot likely died in the Bergen-Belsen concentration camp at least a month earlier than previously believed. The researchers examined archives of the Red Cross, the International Training Service, and the Bergen-Belsen Memorial, as well as testimonies from survivors. Anne and Margot were separated from their mother and sent to work as slave labor in the camp, where they lived in overcrowded conditions with little water, sanitation, or food. The sisters contracted typhus, which was rampant in the camp, and are believed to have died before February 7, 2022. The exact dates of their deaths remain unclear, but witness accounts suggest that they had symptoms of the disease before then. The discovery has led to a reevaluation of the events surrounding Anne Frank's death and has prompted the Anne Frank House to issue a corrected statement regarding the da

In [4]:

import pandas as pd
import numpy as np
from collections import Counter

# Load the validated dataset
df = pd.read_parquet(
    r"C:\Users\yukta ghara\RAGGuard_Project\processed\ragguard_merged.parquet"
)

total_annotations = 0
valid_annotations = 0
invalid_annotations = []

label_counts = Counter()
annotation_records = []

# Check every response and every annotation
for _, row in df.iterrows():

    response = row["response"]
    labels = row["labels"]

    if labels is None:
        continue

    for annotation in labels:

        total_annotations += 1

        start = annotation.get("start")
        end = annotation.get("end")
        annotated_text = annotation.get("text")
        label_type = annotation.get("label_type")

        label_counts[label_type] += 1

        # Validate offset types
        offsets_are_integers = (
            isinstance(start, (int, np.integer))
            and isinstance(end, (int, np.integer))
        )

        if not offsets_are_integers:
            is_valid = False
            extracted_text = None

        elif start < 0 or end < start or end > len(response):
            is_valid = False
            extracted_text = None

        else:
            # Test end-exclusive character offsets
            extracted_text = response[start:end]
            is_valid = extracted_text == annotated_text

        if is_valid:
            valid_annotations += 1
        else:
            invalid_annotations.append({
                "response_id": row["id"],
                "source_id": row["source_id"],
                "start": start,
                "end": end,
                "annotation_text": annotated_text,
                "extracted_text": extracted_text,
                "label_type": label_type
            })

        annotation_records.append({
            "response_id": row["id"],
            "source_id": row["source_id"],
            "start": start,
            "end": end,
            "text": annotated_text,
            "label_type": label_type,
            "is_valid": is_valid
        })


# Results
print("=" * 60)
print("ANNOTATION OFFSET VALIDATION")
print("=" * 60)

print("TOTAL ANNOTATIONS:", total_annotations)
print("VALID OFFSETS:", valid_annotations)
print("INVALID OFFSETS:", len(invalid_annotations))

if total_annotations:
    validity = valid_annotations / total_annotations * 100
    print("VALIDITY:", round(validity, 2), "%")

print("\nANNOTATION TYPE COUNTS:")
for label, count in label_counts.items():
    print(f"{label}: {count}")

print("\nEXAMPLE INVALID ANNOTATIONS:")
for item in invalid_annotations[:5]:
    print(item)

# Save audit results
audit_df = pd.DataFrame(annotation_records)

audit_path = (
    r"C:\Users\yukta ghara\RAGGuard_Project"
    r"\processed\annotation_offset_audit.csv"
)

audit_df.to_csv(audit_path, index=False)

print("\nAudit saved to:", audit_path)

if len(invalid_annotations) == 0:
    print("\nALL ANNOTATION OFFSETS ARE VALID!")


ANNOTATION OFFSET VALIDATION
TOTAL ANNOTATIONS: 14289
VALID OFFSETS: 14289
INVALID OFFSETS: 0
VALIDITY: 100.0 %

ANNOTATION TYPE COUNTS:
Evident Conflict: 5324
Evident Baseless Info: 6237
Subtle Baseless Info: 2527
Subtle Conflict: 201

EXAMPLE INVALID ANNOTATIONS:

Audit saved to: C:\Users\yukta ghara\RAGGuard_Project\processed\annotation_offset_audit.csv

ALL ANNOTATION OFFSETS ARE VALID!


In [5]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert/distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

print("TOKENIZER LOADED SUCCESSFULLY")
print("Tokenizer:", MODEL_NAME)
print("Fast tokenizer:", tokenizer.is_fast)
print("Vocabulary size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

C:\Users\yukta ghara\Documents\Python Scripts\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\yukta ghara\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

TOKENIZER LOADED SUCCESSFULLY
Tokenizer: distilbert/distilbert-base-uncased
Fast tokenizer: True
Vocabulary size: 30522


In [6]:
# Use the same example response we inspected earlier
sample = df[df["id"] == 2].iloc[0]

encoding = tokenizer(
    sample["response"],
    return_offsets_mapping=True,
    add_special_tokens=True,
    truncation=False
)

tokens = tokenizer.convert_ids_to_tokens(
    encoding["input_ids"]
)

offsets = encoding["offset_mapping"]

print("TOTAL TOKENS:", len(tokens))
print("\nFIRST 30 TOKENS AND OFFSETS:\n")

for token, offset in list(zip(tokens, offsets))[:30]:
    print(f"{token:20} {offset}")

TOTAL TOKENS: 192

FIRST 30 TOKENS AND OFFSETS:

[CLS]                (0, 0)
new                  (0, 3)
research             (4, 12)
conducted            (13, 22)
by                   (23, 25)
the                  (26, 29)
anne                 (30, 34)
frank                (35, 40)
house                (41, 46)
has                  (47, 50)
revealed             (51, 59)
that                 (60, 64)
anne                 (65, 69)
frank                (70, 75)
and                  (76, 79)
her                  (80, 83)
sister               (84, 90)
margot               (91, 97)
likely               (98, 104)
died                 (105, 109)
in                   (110, 112)
the                  (113, 116)
bergen               (117, 123)
-                    (123, 124)
bel                  (124, 127)
##sen                (127, 130)
concentration        (131, 144)
camp                 (145, 149)
at                   (150, 152)
least                (153, 158)


In [9]:

final_input_ids = tokenizer.build_inputs_with_special_tokens(
    chunk_input_ids
)

final_labels = (
    [-100]
    + chunk_labels.tolist()
    + [-100]
)

AttributeError: BertTokenizer has no attribute build_inputs_with_special_tokens

In [8]:

xample["attention_mask"])

    assert example["token_labels"][0] == -100
    assert example["token_labels"][-1] == -100


# ---------------------------------------------------------
# 6. Print results
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("RAGGUARD CHUNKING RESULTS")
print("=" * 60)

print("Original responses:", len(df))
print("Total original tokens:", total_original_tokens)
print("Total preserved content tokens:", total_chunked_tokens)

print("Total training chunks:", len(chunks_df))

print("Maximum sequence length:", sequence_lengths.max())
print("Minimum sequence length:", sequence_lengths.min())

print("\nOriginal split distribution:")
print(chunks_df["split"].value_counts().to_string())

print("\nChunks by task type:")
print(chunks_df["task_type"].value_counts().to_string())

print("\nTotal hallucinated tokens:",
      total_hallucinated_tokens)

print("\nChunks exceeding 512 tokens:",
      int((sequence_lengths > MAX_LENGTH).sum()))

print("\nToken preservation:",
      total_original_tokens == total_chunked_tokens)


# ---------------------------------------------------------
# 7. Save chunked dataset
# ---------------------------------------------------------

output_path = (
    r"C:\Users\yukta ghara\RAGGuard_Project"
    r"\processed\ragguard_tokenized_chunks.parquet"
)

chunks_df.to_parquet(output_path, index=False)

print("\nSaved chunked dataset to:")
print(output_path)

print("\nSTAGE 2D COMPLETED SUCCESSFULLY!")


Maximum model sequence length: 512
Maximum content tokens per chunk: 510


AttributeError: BertTokenizer has no attribute build_inputs_with_special_tokens

In [16]:
final_input_ids = (
    [tokenizer.cls_token_id]
    + chunk_input_ids
    + [tokenizer.sep_token_id]
)

final_labels = (
    [-100]
    + chunk_labels.tolist()
    + [-100]
)

In [13]:
print("CLS token:", tokenizer.cls_token)
print("CLS token ID:", tokenizer.cls_token_id)

print("SEP token:", tokenizer.sep_token)
print("SEP token ID:", tokenizer.sep_token_id)

assert tokenizer.cls_token_id is not None
assert tokenizer.sep_token_id is not None

print("\nSPECIAL TOKEN VALIDATION SUCCESSFUL!")

CLS token: [CLS]
CLS token ID: 101
SEP token: [SEP]
SEP token ID: 102

SPECIAL TOKEN VALIDATION SUCCESSFUL!


In [14]:
training_examples = []

In [15]:
final_input_ids = (
    [tokenizer.cls_token_id]
    + chunk_input_ids
    + [tokenizer.sep_token_id]
)

final_labels = (
    [-100]
    + chunk_labels.tolist()
    + [-100]
)

In [17]:

import pandas as pd
import numpy as np

# 1. Configure DistilBERT sequence length
MAX_LENGTH = 512
CONTENT_LENGTH = MAX_LENGTH - tokenizer.num_special_tokens_to_add(pair=False)

print("Maximum model sequence length:", MAX_LENGTH)
print("Maximum content tokens per chunk:", CONTENT_LENGTH)

# 2. Load the merged dataset
df = pd.read_parquet(
    r"C:\Users\yukta ghara\RAGGuard_Project\processed\ragguard_merged.parquet"
)

# 3. Initialize storage
training_examples = []

total_original_tokens = 0
total_chunked_tokens = 0
total_hallucinated_tokens = 0

# 4. Process every response
for _, row in df.iterrows():

    response = row["response"]
    annotations = row["labels"]

    encoding = tokenizer(
        response,
        return_offsets_mapping=True,
        add_special_tokens=False,
        truncation=False
    )

    input_ids = encoding["input_ids"]
    offsets = encoding["offset_mapping"]

    # Initially mark every content token as supported (0)
    token_labels = np.zeros(len(input_ids), dtype=np.int64)

    # Mark hallucinated tokens (1)
    for annotation in annotations:

        start = annotation["start"]
        end = annotation["end"]

        for token_index, (token_start, token_end) in enumerate(offsets):

            if token_start == token_end:
                continue

            if token_start < end and token_end > start:
                token_labels[token_index] = 1

    total_original_tokens += len(input_ids)
    total_hallucinated_tokens += int(token_labels.sum())

    # 5. Split long responses into chunks
    for chunk_number, chunk_start in enumerate(
        range(0, len(input_ids), CONTENT_LENGTH)
    ):

        chunk_end = min(
            chunk_start + CONTENT_LENGTH,
            len(input_ids)
        )

        chunk_input_ids = input_ids[chunk_start:chunk_end]
        chunk_labels = token_labels[chunk_start:chunk_end]

        # Add CLS and SEP manually
        final_input_ids = (
            [tokenizer.cls_token_id]
            + chunk_input_ids
            + [tokenizer.sep_token_id]
        )

        final_labels = (
            [-100]
            + chunk_labels.tolist()
            + [-100]
        )

        training_examples.append({
            "response_id": row["id"],
            "source_id": row["source_id"],
            "model": row["model"],
            "split": row["split"],
            "task_type": row["task_type"],
            "chunk_number": chunk_number,
            "input_ids": final_input_ids,
            "attention_mask": [1] * len(final_input_ids),
            "token_labels": final_labels,
            "content_token_count": len(chunk_input_ids),
            "hallucinated_token_count": int(chunk_labels.sum())
        })

        total_chunked_tokens += len(chunk_input_ids)

# 6. Convert to DataFrame
chunks_df = pd.DataFrame(training_examples)

sequence_lengths = chunks_df["input_ids"].apply(len)

# 7. Validate the results
assert sequence_lengths.max() <= MAX_LENGTH
assert total_original_tokens == total_chunked_tokens

for _, example in chunks_df.iterrows():

    assert len(example["input_ids"]) == len(example["token_labels"])
    assert len(example["input_ids"]) == len(example["attention_mask"])

    assert example["token_labels"][0] == -100
    assert example["token_labels"][-1] == -100

# 8. Display results
print("\n" + "=" * 55)
print("RAGGUARD CHUNKING RESULTS")
print("=" * 55)

print("Original responses:", len(df))
print("Total original content tokens:", total_original_tokens)
print("Total preserved content tokens:", total_chunked_tokens)
print("Total training chunks:", len(chunks_df))

print("Maximum sequence length:", sequence_lengths.max())
print("Minimum sequence length:", sequence_lengths.min())

print("\nOriginal split distribution:")
print(chunks_df["split"].value_counts().to_string())

print("\nChunks by task type:")
print(chunks_df["task_type"].value_counts().to_string())

print("\nTotal hallucinated tokens:", total_hallucinated_tokens)

print(
    "\nChunks exceeding 512 tokens:",
    int((sequence_lengths > MAX_LENGTH).sum())
)

print(
    "\nToken preservation:",
    total_original_tokens == total_chunked_tokens
)

# 9. Save the chunked dataset
output_path = (
    r"C:\Users\yukta ghara\RAGGuard_Project"
    r"\processed\ragguard_tokenized_chunks.parquet"
)

chunks_df.to_parquet(output_path, index=False)

print("\nSaved chunked dataset to:")
print(output_path)

print("\nSTAGE 2D COMPLETED SUCCESSFULLY!")


Maximum model sequence length: 512
Maximum content tokens per chunk: 510

RAGGUARD CHUNKING RESULTS
Original responses: 17790
Total original content tokens: 2940531
Total preserved content tokens: 2940531
Total training chunks: 17791
Maximum sequence length: 512
Minimum sequence length: 3

Original split distribution:
split
train    15091
test      2700

Chunks by task type:
task_type
Data2txt    6198
QA          5934
Summary     5659

Total hallucinated tokens: 158837

Chunks exceeding 512 tokens: 0

Token preservation: True

Saved chunked dataset to:
C:\Users\yukta ghara\RAGGuard_Project\processed\ragguard_tokenized_chunks.parquet

STAGE 2D COMPLETED SUCCESSFULLY!


In [18]:

import pandas as pd

chunks_df = pd.read_parquet(
    r"C:\Users\yukta ghara\RAGGuard_Project\processed\ragguard_tokenized_chunks.parquet"
)

print("TOTAL CHUNKS:", len(chunks_df))

print("\nUNIQUE SOURCE IDS:")
print(chunks_df.groupby("split")["source_id"].nunique())

print("\nUNIQUE RESPONSE IDS:")
print(chunks_df.groupby("split")["response_id"].nunique())

print("\nCHUNKS PER SPLIT:")
print(chunks_df["split"].value_counts())

print("\nSOURCE ID OVERLAP BETWEEN TRAIN AND TEST:")

train_sources = set(
    chunks_df.loc[chunks_df["split"] == "train", "source_id"]
)

test_sources = set(
    chunks_df.loc[chunks_df["split"] == "test", "source_id"]
)

print("Overlapping source IDs:", len(train_sources & test_sources))

print("\nSTAGE 3 INITIAL AUDIT COMPLETED!")


TOTAL CHUNKS: 17791

UNIQUE SOURCE IDS:
split
test      450
train    2515
Name: source_id, dtype: int64

UNIQUE RESPONSE IDS:
split
test      2700
train    15090
Name: response_id, dtype: int64

CHUNKS PER SPLIT:
split
train    15091
test      2700
Name: count, dtype: int64

SOURCE ID OVERLAP BETWEEN TRAIN AND TEST:
Overlapping source IDs: 0

STAGE 3 INITIAL AUDIT COMPLETED!


In [19]:

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# Load chunked dataset
chunks_df = pd.read_parquet(
    r"C:\Users\yukta ghara\RAGGuard_Project\processed\ragguard_tokenized_chunks.parquet"
)

# Separate original training and test data
original_train = chunks_df[chunks_df["split"] == "train"].copy()
test_df = chunks_df[chunks_df["split"] == "test"].copy()

# Split training data by source_id, not by individual chunks
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.10,
    random_state=42
)

train_indices, val_indices = next(
    splitter.split(
        original_train,
        groups=original_train["source_id"]
    )
)

train_df = original_train.iloc[train_indices].copy()
val_df = original_train.iloc[val_indices].copy()

# Assign final split labels
train_df["split"] = "train"
val_df["split"] = "validation"
test_df["split"] = "test"

# Combine all three datasets
final_df = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

# Verify source-level separation
train_sources = set(train_df["source_id"])
val_sources = set(val_df["source_id"])
test_sources = set(test_df["source_id"])

print("TRAIN SOURCE ARTICLES:", len(train_sources))
print("VALIDATION SOURCE ARTICLES:", len(val_sources))
print("TEST SOURCE ARTICLES:", len(test_sources))

print("\nSOURCE OVERLAP CHECK:")
print("Train vs Validation:", len(train_sources & val_sources))
print("Train vs Test:", len(train_sources & test_sources))
print("Validation vs Test:", len(val_sources & test_sources))

print("\nCHUNK DISTRIBUTION:")
print(final_df["split"].value_counts())

print("\nRESPONSE DISTRIBUTION:")
print(
    final_df.groupby("split")["response_id"].nunique()
)

# Save final split dataset
output_path = (
    r"C:\Users\yukta ghara\RAGGuard_Project"
    r"\processed\ragguard_final_splits.parquet"
)

final_df.to_parquet(output_path, index=False)

assert len(train_sources & val_sources) == 0
assert len(train_sources & test_sources) == 0
assert len(val_sources & test_sources) == 0

print("\nSTAGE 3A COMPLETED SUCCESSFULLY!")
print("Saved:", output_path)


TRAIN SOURCE ARTICLES: 2263
VALIDATION SOURCE ARTICLES: 252
TEST SOURCE ARTICLES: 450

SOURCE OVERLAP CHECK:
Train vs Validation: 0
Train vs Test: 0
Validation vs Test: 0

CHUNK DISTRIBUTION:
split
train         13579
test           2700
validation     1512
Name: count, dtype: int64

RESPONSE DISTRIBUTION:
split
test           2700
train         13578
validation     1512
Name: response_id, dtype: int64

STAGE 3A COMPLETED SUCCESSFULLY!
Saved: C:\Users\yukta ghara\RAGGuard_Project\processed\ragguard_final_splits.parquet


In [20]:

import pandas as pd
import numpy as np

# Load final dataset
final_df = pd.read_parquet(
    r"C:\Users\yukta ghara\RAGGuard_Project\processed\ragguard_final_splits.parquet"
)

print("DATASET SHAPE:", final_df.shape)

# Count token labels, excluding special tokens (-100)
for split_name in ["train", "validation", "test"]:

    subset = final_df[final_df["split"] == split_name]

    supported = 0
    hallucinated = 0
    ignored = 0

    for labels in subset["token_labels"]:

        labels = np.array(labels)

        supported += int((labels == 0).sum())
        hallucinated += int((labels == 1).sum())
        ignored += int((labels == -100).sum())

    total = supported + hallucinated

    print("\n" + "=" * 45)
    print(split_name.upper())

    print("Chunks:", len(subset))
    print("Supported tokens:", supported)
    print("Hallucinated tokens:", hallucinated)
    print("Ignored special tokens:", ignored)

    if total > 0:
        print(
            "Hallucinated token percentage:",
            round(hallucinated / total * 100, 2),
            "%"
        )

print("\nSTAGE 3B INITIAL AUDIT COMPLETED!")


DATASET SHAPE: (17791, 11)

TRAIN
Chunks: 13579
Supported tokens: 2137150
Hallucinated tokens: 126547
Ignored special tokens: 27158
Hallucinated token percentage: 5.59 %

VALIDATION
Chunks: 1512
Supported tokens: 232070
Hallucinated tokens: 14516
Ignored special tokens: 3024
Hallucinated token percentage: 5.89 %

TEST
Chunks: 2700
Supported tokens: 412474
Hallucinated tokens: 17774
Ignored special tokens: 5400
Hallucinated token percentage: 4.13 %

STAGE 3B INITIAL AUDIT COMPLETED!


In [21]:

import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# Collect training labels only
train_df = final_df[final_df["split"] == "train"]

all_train_labels = []

for labels in train_df["token_labels"]:
    all_train_labels.extend(
        [label for label in labels if label != -100]
    )

all_train_labels = np.array(all_train_labels)

# Calculate balanced class weights
classes = np.array([0, 1])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=all_train_labels
)

print("CLASS WEIGHTS")
print("Supported (0):", class_weights[0])
print("Hallucinated (1):", class_weights[1])

print("\nCLASS COUNTS")
print("Supported:", np.sum(all_train_labels == 0))
print("Hallucinated:", np.sum(all_train_labels == 1))

print("\nSTAGE 3C COMPLETED SUCCESSFULLY!")


CLASS WEIGHTS
Supported (0): 0.5296064852724423
Hallucinated (1): 8.944095869518835

CLASS COUNTS
Supported: 2137150
Hallucinated: 126547

STAGE 3C COMPLETED SUCCESSFULLY!


In [22]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd

In [24]:

import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader

# Load the final dataset
final_df = pd.read_parquet(
    r"C:\Users\yukta ghara\RAGGuard_Project\processed\ragguard_final_splits.parquet"
)

# Define PyTorch Dataset
class RagGuardDataset(Dataset):

    def __init__(self, dataframe):
        self.data = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]

        return {
            "input_ids": torch.tensor(
                row["input_ids"], dtype=torch.long
            ),
            "attention_mask": torch.tensor(
                row["attention_mask"], dtype=torch.long
            ),
            "labels": torch.tensor(
                row["token_labels"], dtype=torch.long
            )
        }

# Create datasets
train_data = RagGuardDataset(
    final_df[final_df["split"] == "train"]
)

val_data = RagGuardDataset(
    final_df[final_df["split"] == "validation"]
)

test_data = RagGuardDataset(
    final_df[final_df["split"] == "test"]
)

# Create DataLoaders
BATCH_SIZE = 4

train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_data,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_data,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Display results
print("STAGE 4 RESULTS")
print("Training examples:", len(train_data))
print("Validation examples:", len(val_data))
print("Testing examples:", len(test_data))

print("\nTraining batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

# Inspect one example
sample = train_data[0]

print("\nSAMPLE TENSOR SHAPES")
print("Input IDs:", sample["input_ids"].shape)
print("Attention Mask:", sample["attention_mask"].shape)
print("Labels:", sample["labels"].shape)

print("\nSTAGE 4 COMPLETED SUCCESSFULLY!")



STAGE 4 RESULTS
Training examples: 13579
Validation examples: 1512
Testing examples: 2700

Training batches: 3395
Validation batches: 378
Testing batches: 675

SAMPLE TENSOR SHAPES
Input IDs: torch.Size([92])
Attention Mask: torch.Size([92])
Labels: torch.Size([92])

STAGE 4 COMPLETED SUCCESSFULLY!


In [25]:

import torch
from transformers import (
    AutoModelForTokenClassification,
    AutoConfig
)

MODEL_NAME = "distilbert/distilbert-base-uncased"

# Configure model for binary token classification
config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={
        0: "SUPPORTED",
        1: "HALLUCINATED"
    },
    label2id={
        "SUPPORTED": 0,
        "HALLUCINATED": 1
    }
)

# Load pretrained DistilBERT
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    config=config
)

# CPU device
device = torch.device("cpu")
model.to(device)

print("MODEL LOADED SUCCESSFULLY!")

print("\nModel:", MODEL_NAME)
print("Number of labels:", model.config.num_labels)
print("Label mapping:", model.config.id2label)
print("Device:", device)

# Check model parameters
total_parameters = sum(
    parameter.numel() for parameter in model.parameters()
)

print("\nTotal model parameters:", total_parameters)

print("\nSTAGE 5 COMPLETED SUCCESSFULLY!")


C:\Users\yukta ghara\Documents\Python Scripts\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\yukta ghara\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


MODEL LOADED SUCCESSFULLY!

Model: distilbert/distilbert-base-uncased
Number of labels: 2
Label mapping: {0: 'SUPPORTED', 1: 'HALLUCINATED'}
Device: cpu

Total model parameters: 66364418

STAGE 5 COMPLETED SUCCESSFULLY!


In [26]:

import torch
import torch.nn as nn

# Class weights calculated from training data
class_weights_tensor = torch.tensor(
    [0.5296064852724423, 8.944095869518835],
    dtype=torch.float32
)

# Move weights to CPU
class_weights_tensor = class_weights_tensor.to(device)

# Weighted cross-entropy loss
criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor,
    ignore_index=-100
)

print("WEIGHTED LOSS CONFIGURATION")

print("Supported weight:", class_weights_tensor[0].item())
print("Hallucinated weight:", class_weights_tensor[1].item())

print("Ignored label:", -100)

print("\nLoss function:", criterion)

print("\nSTAGE 6 COMPLETED SUCCESSFULLY!")


WEIGHTED LOSS CONFIGURATION
Supported weight: 0.5296064615249634
Hallucinated weight: 8.944095611572266
Ignored label: -100

Loss function: CrossEntropyLoss()

STAGE 6 COMPLETED SUCCESSFULLY!


In [27]:

from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader

# Padding function for variable-length sequences
def ragguard_collate_fn(batch):

    input_ids = [item["input_ids"] for item in batch]
    attention_masks = [item["attention_mask"] for item in batch]
    labels = [item["labels"] for item in batch]

    # Pad input IDs using DistilBERT's padding token
    input_ids = pad_sequence(
        input_ids,
        batch_first=True,
        padding_value=tokenizer.pad_token_id
    )

    # Pad attention masks with 0
    attention_masks = pad_sequence(
        attention_masks,
        batch_first=True,
        padding_value=0
    )

    # Ignore padded positions in the loss
    labels = pad_sequence(
        labels,
        batch_first=True,
        padding_value=-100
    )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_masks,
        "labels": labels
    }


# Rebuild DataLoaders
BATCH_SIZE = 4

train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=ragguard_collate_fn
)

val_loader = DataLoader(
    val_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=ragguard_collate_fn
)

test_loader = DataLoader(
    test_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=ragguard_collate_fn
)

# Inspect one batch
batch = next(iter(train_loader))

print("DYNAMIC PADDING VALIDATION")

print("Input IDs shape:", batch["input_ids"].shape)
print("Attention mask shape:", batch["attention_mask"].shape)
print("Labels shape:", batch["labels"].shape)

print("\nMaximum batch sequence length:",
      batch["input_ids"].shape[1])

print("\nPadding token ID:", tokenizer.pad_token_id)

print("\nSTAGE 7A COMPLETED SUCCESSFULLY!")


DYNAMIC PADDING VALIDATION
Input IDs shape: torch.Size([4, 262])
Attention mask shape: torch.Size([4, 262])
Labels shape: torch.Size([4, 262])

Maximum batch sequence length: 262

Padding token ID: 0

STAGE 7A COMPLETED SUCCESSFULLY!


In [28]:

from torch.optim import AdamW

# Training configuration
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

# Initialize optimizer
optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print("OPTIMIZER CONFIGURATION")

print("Optimizer:", type(optimizer).__name__)
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)

print("\nSTAGE 7B COMPLETED SUCCESSFULLY!")


OPTIMIZER CONFIGURATION
Optimizer: AdamW
Learning rate: 2e-05
Weight decay: 0.01
Epochs: 3
Batch size: 4

STAGE 7B COMPLETED SUCCESSFULLY!


In [29]:

# Get one training batch
batch = next(iter(train_loader))

# Move batch to CPU
input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["labels"].to(device)

# Clear previous gradients
optimizer.zero_grad()

# Forward pass
outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask
)

logits = outputs.logits

print("MODEL OUTPUT SHAPE:", logits.shape)

# Calculate weighted loss
loss = criterion(
    logits.view(-1, model.config.num_labels),
    labels.view(-1)
)

print("TEST LOSS:", loss.item())

# Backward pass
loss.backward()

print("BACKPROPAGATION: SUCCESSFUL")

# Optimizer update
optimizer.step()

print("OPTIMIZER UPDATE: SUCCESSFUL")

print("\nSTAGE 7C COMPLETED SUCCESSFULLY!")


MODEL OUTPUT SHAPE: torch.Size([4, 351, 2])
TEST LOSS: 0.6731420755386353
BACKPROPAGATION: SUCCESSFUL
OPTIMIZER UPDATE: SUCCESSFUL

STAGE 7C COMPLETED SUCCESSFULLY!


In [30]:

import os
import time
import torch

# Train for one epoch
EPOCHS = 1

# Track training time
start_time = time.time()

# Store epoch losses
training_losses = []
validation_losses = []

print("STARTING RAGGUARD TRAINING")
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Epochs:", EPOCHS)

# ==========================================
# TRAINING
# ==========================================

for epoch in range(EPOCHS):

    model.train()

    total_train_loss = 0.0
    total_train_weight = 0.0

    print(f"\nEPOCH {epoch + 1}/{EPOCHS}")
    print("Training started...")

    for step, batch in enumerate(train_loader):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits

        # Weighted loss
        loss = criterion(
            logits.reshape(-1, model.config.num_labels),
            labels.reshape(-1)
        )

        # Backpropagation
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=1.0
        )

        # Update weights
        optimizer.step()

        # Accumulate correctly weighted loss
        valid_labels = labels[labels != -100]

        batch_weight = class_weights_tensor[
            valid_labels
        ].sum().item()

        total_train_loss += loss.item() * batch_weight
        total_train_weight += batch_weight

        # Progress update
        if (step + 1) % 200 == 0 or (step + 1) == len(train_loader):
            print(
                f"Training batch {step + 1}/{len(train_loader)}"
            )

    train_loss = total_train_loss / total_train_weight
    training_losses.append(train_loss)

    print(f"\nTraining Loss: {train_loss:.4f}")

    # ==========================================
    # VALIDATION
    # ==========================================

    model.eval()

    total_val_loss = 0.0
    total_val_weight = 0.0

    print("\nValidation started...")

    with torch.no_grad():

        for batch in val_loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            logits = outputs.logits

            loss = criterion(
                logits.reshape(-1, model.config.num_labels),
                labels.reshape(-1)
            )

            valid_labels = labels[labels != -100]

            batch_weight = class_weights_tensor[
                valid_labels
            ].sum().item()

            total_val_loss += loss.item() * batch_weight
            total_val_weight += batch_weight

    val_loss = total_val_loss / total_val_weight
    validation_losses.append(val_loss)

    print(f"Validation Loss: {val_loss:.4f}")

# ==========================================
# SAVE CHECKPOINT
# ==========================================

model_dir = (
    r"C:\Users\yukta ghara\RAGGuard_Project\models"
)

os.makedirs(model_dir, exist_ok=True)

checkpoint_path = os.path.join(
    model_dir, "ragguard_distilbert_epoch1.pt"
)

torch.save(
    {
        "epoch": EPOCHS,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "training_loss": training_losses,
        "validation_loss": validation_losses
    },
    checkpoint_path
)

# ==========================================
# FINAL RESULTS
# ==========================================

elapsed_time = (time.time() - start_time) / 60

print("\n" + "=" * 45)
print("RAGGUARD TRAINING COMPLETED")
print("=" * 45)

print("Training Loss:", training_losses[-1])
print("Validation Loss:", validation_losses[-1])
print(f"Training Time: {elapsed_time:.2f} minutes")
print("Checkpoint:", checkpoint_path)

print("\nSTAGE 8A COMPLETED SUCCESSFULLY!")


STARTING RAGGUARD TRAINING
Training batches: 3395
Validation batches: 378
Epochs: 1

EPOCH 1/1
Training started...
Training batch 200/3395
Training batch 400/3395
Training batch 600/3395
Training batch 800/3395
Training batch 1000/3395
Training batch 1200/3395
Training batch 1400/3395
Training batch 1600/3395
Training batch 1800/3395
Training batch 2000/3395
Training batch 2200/3395
Training batch 2400/3395
Training batch 2600/3395
Training batch 2800/3395
Training batch 3000/3395
Training batch 3200/3395
Training batch 3395/3395

Training Loss: 0.6367

Validation started...
Validation Loss: 0.6177

RAGGUARD TRAINING COMPLETED
Training Loss: 0.6366512133566132
Validation Loss: 0.6176978299420745
Training Time: 187.80 minutes
Checkpoint: C:\Users\yukta ghara\RAGGuard_Project\models\ragguard_distilbert_epoch1.pt

STAGE 8A COMPLETED SUCCESSFULLY!


In [31]:

import os
import torch

checkpoint_path = (
    r"C:\Users\yukta ghara\RAGGuard_Project\models"
    r"\ragguard_distilbert_epoch1.pt"
)

print("CHECKPOINT VERIFICATION")

print("File exists:", os.path.exists(checkpoint_path))

if os.path.exists(checkpoint_path):
    print(
        "File size:",
        round(os.path.getsize(checkpoint_path) / (1024 * 1024), 2),
        "MB"
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=True
    )

    print("Saved epoch:", checkpoint["epoch"])
    print("Training loss:", checkpoint["training_loss"])
    print("Validation loss:", checkpoint["validation_loss"])

    print("\nSTAGE 9 PREPARATION SUCCESSFUL!")
else:
    print("ERROR: Checkpoint file not found.")


CHECKPOINT VERIFICATION
File exists: True
File size: 759.61 MB
Saved epoch: 1
Training loss: [0.6366512133566132]
Validation loss: [0.6176978299420745]

STAGE 9 PREPARATION SUCCESSFUL!


In [32]:

import torch
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# ==========================================
# 1. LOAD SAVED MODEL
# ==========================================

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=True
)

model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()

print("SAVED RAGGUARD MODEL LOADED SUCCESSFULLY!")

# ==========================================
# 2. TEST EVALUATION
# ==========================================

all_predictions = []
all_true_labels = []

print("\nSTARTING TEST SET EVALUATION")

with torch.no_grad():

    for step, batch in enumerate(test_loader):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(
            outputs.logits,
            dim=-1
        )

        # Exclude special tokens and padding
        valid_mask = labels != -100

        all_predictions.extend(
            predictions[valid_mask].cpu().numpy()
        )

        all_true_labels.extend(
            labels[valid_mask].cpu().numpy()
        )

        if (step + 1) % 100 == 0 or (step + 1) == len(test_loader):
            print(
                f"Evaluated batch {step + 1}/{len(test_loader)}"
            )

# ==========================================
# 3. CALCULATE METRICS
# ==========================================

y_true = np.array(all_true_labels)
y_pred = np.array(all_predictions)

accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true, y_pred, pos_label=1, zero_division=0
)

recall = recall_score(
    y_true, y_pred, pos_label=1, zero_division=0
)

f1 = f1_score(
    y_true, y_pred, pos_label=1, zero_division=0
)

cm = confusion_matrix(
    y_true, y_pred, labels=[0, 1]
)

# ==========================================
# 4. DISPLAY RESULTS
# ==========================================

print("\n" + "=" * 50)
print("RAGGUARD TEST SET EVALUATION")
print("=" * 50)

print("Total evaluated tokens:", len(y_true))
print("Accuracy:", round(accuracy, 4))
print("Hallucination Precision:", round(precision, 4))
print("Hallucination Recall:", round(recall, 4))
print("Hallucination F1-score:", round(f1, 4))

print("\nCONFUSION MATRIX")
print("Rows = Actual, Columns = Predicted")
print("Labels: 0 = Supported, 1 = Hallucinated")
print(cm)

print("\nCLASSIFICATION REPORT")
print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=["SUPPORTED", "HALLUCINATED"],
        zero_division=0
    )
)

print("\nSTAGE 9B COMPLETED SUCCESSFULLY!")


SAVED RAGGUARD MODEL LOADED SUCCESSFULLY!

STARTING TEST SET EVALUATION
Evaluated batch 100/675
Evaluated batch 200/675
Evaluated batch 300/675
Evaluated batch 400/675
Evaluated batch 500/675
Evaluated batch 600/675
Evaluated batch 675/675

RAGGUARD TEST SET EVALUATION
Total evaluated tokens: 430248
Accuracy: 0.8543
Hallucination Precision: 0.151
Hallucination Recall: 0.5467
Hallucination F1-score: 0.2367

CONFUSION MATRIX
Rows = Actual, Columns = Predicted
Labels: 0 = Supported, 1 = Hallucinated
[[357851  54623]
 [  8057   9717]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

   SUPPORTED       0.98      0.87      0.92    412474
HALLUCINATED       0.15      0.55      0.24     17774

    accuracy                           0.85    430248
   macro avg       0.56      0.71      0.58    430248
weighted avg       0.94      0.85      0.89    430248


STAGE 9B COMPLETED SUCCESSFULLY!


In [33]:

import torch
import numpy as np

from sklearn.metrics import precision_recall_curve

model.eval()

validation_probabilities = []
validation_true_labels = []

print("COLLECTING VALIDATION PREDICTIONS")

with torch.no_grad():

    for step, batch in enumerate(val_loader):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Convert logits into probabilities
        probabilities = torch.softmax(
            outputs.logits,
            dim=-1
        )

        # Probability of hallucination (class 1)
        hallucination_probs = probabilities[:, :, 1]

        # Ignore padding and special tokens
        valid_mask = labels != -100

        validation_probabilities.extend(
            hallucination_probs[valid_mask].cpu().numpy()
        )

        validation_true_labels.extend(
            labels[valid_mask].cpu().numpy()
        )

        if (step + 1) % 100 == 0 or (step + 1) == len(val_loader):
            print(
                f"Validation batch {step + 1}/{len(val_loader)}"
            )

# Convert to NumPy arrays
val_probs = np.array(validation_probabilities)
val_true = np.array(validation_true_labels)

print("\nVALIDATION PREDICTION RESULTS")
print("Total tokens:", len(val_true))
print("Supported tokens:", np.sum(val_true == 0))
print("Hallucinated tokens:", np.sum(val_true == 1))
print("Probability range:", val_probs.min(), "to", val_probs.max())

print("\nSTAGE 9C DATA COLLECTION COMPLETED!")


COLLECTING VALIDATION PREDICTIONS
Validation batch 100/378
Validation batch 200/378
Validation batch 300/378
Validation batch 378/378

VALIDATION PREDICTION RESULTS
Total tokens: 246586
Supported tokens: 232070
Hallucinated tokens: 14516
Probability range: 0.0017576757 to 0.9917972

STAGE 9C DATA COLLECTION COMPLETED!


In [34]:

import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

# Thresholds to evaluate
thresholds = np.arange(0.10, 0.91, 0.05)

results = []

for threshold in thresholds:

    predictions = (
        val_probs >= threshold
    ).astype(int)

    precision = precision_score(
        val_true,
        predictions,
        pos_label=1,
        zero_division=0
    )

    recall = recall_score(
        val_true,
        predictions,
        pos_label=1,
        zero_division=0
    )

    f1 = f1_score(
        val_true,
        predictions,
        pos_label=1,
        zero_division=0
    )

    results.append({
        "Threshold": round(float(threshold), 2),
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1
    })

# Create results table
threshold_df = pd.DataFrame(results)

# Display all results
print("THRESHOLD CALIBRATION RESULTS")
print(
    threshold_df.to_string(
        index=False,
        formatters={
            "Precision": "{:.4f}".format,
            "Recall": "{:.4f}".format,
            "F1-score": "{:.4f}".format
        }
    )
)

# Find best threshold using validation F1
best_index = threshold_df["F1-score"].idxmax()

BEST_THRESHOLD = float(
    threshold_df.loc[best_index, "Threshold"]
)

print("\nBEST VALIDATION THRESHOLD:", BEST_THRESHOLD)

print(
    "BEST VALIDATION F1:",
    round(threshold_df.loc[best_index, "F1-score"], 4)
)

print("\nSTAGE 9D COMPLETED SUCCESSFULLY!")


THRESHOLD CALIBRATION RESULTS
 Threshold Precision Recall F1-score
      0.10    0.1035 0.8848   0.1853
      0.15    0.1179 0.8302   0.2065
      0.20    0.1319 0.7881   0.2259
      0.25    0.1448 0.7460   0.2425
      0.30    0.1572 0.7078   0.2573
      0.35    0.1708 0.6754   0.2727
      0.40    0.1840 0.6447   0.2863
      0.45    0.1973 0.6144   0.2987
      0.50    0.2105 0.5850   0.3096
      0.55    0.2246 0.5583   0.3204
      0.60    0.2385 0.5319   0.3294
      0.65    0.2545 0.5068   0.3388
      0.70    0.2716 0.4810   0.3471
      0.75    0.2906 0.4481   0.3526
      0.80    0.3144 0.4098   0.3558
      0.85    0.3443 0.3579   0.3510
      0.90    0.3991 0.2864   0.3335

BEST VALIDATION THRESHOLD: 0.8
BEST VALIDATION F1: 0.3558

STAGE 9D COMPLETED SUCCESSFULLY!


In [ ]:

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# ==========================================
# 1. FIXED VALIDATION THRESHOLD
# ==========================================

BEST_THRESHOLD = 0.80

# ==========================================
# 2. COLLECT TEST PROBABILITIES
# ==========================================

model.eval()

test_probabilities = []
test_true_labels = []

print("COLLECTING TEST PREDICTIONS")

with torch.no_grad():

    for step, batch in enumerate(test_loader):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probabilities = torch.softmax(
            outputs.logits,
            dim=-1
        )

        hallucination_probs = probabilities[:, :, 1]

        valid_mask = labels != -100

        test_probabilities.extend(
            hallucination_probs[valid_mask].cpu().numpy()
        )

        test_true_labels.extend(
            labels[valid_mask].cpu().numpy()
        )

        if (step + 1) % 100 == 0 or (step + 1) == len(test_loader):
            print(
                f"Test batch {step + 1}/{len(test_loader)}"
            )

test_probs = np.array(test_probabilities)
test_true = np.array(test_true_labels)

# ==========================================
# 3. COMPARE TWO THRESHOLDS
# ==========================================

threshold_results = []

for threshold in [0.50, BEST_THRESHOLD]:

    predictions = (
        test_probs >= threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(test_true, predictions),
        "Precision": precision_score(
            test_true, predictions, zero_division=0
        ),
        "Recall": recall_score(
            test_true, predictions, zero_division=0
        ),
        "F1-score": f1_score(
            test_true, predictions, zero_division=0
        )
    })

comparison_df = pd.DataFrame(threshold_results)

print("\nTEST SET THRESHOLD COMPARISON")

print(
    comparison_df.to_string(
        index=False,
        formatters={
            "Accuracy": "{:.4f}".format,
            "Precision": "{:.4f}".format,
            "Recall": "{:.4f}".format,
            "F1-score": "{:.4f}".format
        }
    )
)

# ==========================================
# 4. FINAL CALIBRATED RESULTS
# ==========================================

final_predictions = (
    test_probs >= BEST_THRESHOLD
).astype(int)

print("\nCALIBRATED CONFUSION MATRIX")
print(
    confusion_matrix(
        test_true,
        final_predictions,
        labels=[0, 1]
    )
)

print("\nCALIBRATED CLASSIFICATION REPORT")

print(
    classification_report(
        test_true,
        final_predictions,
        labels=[0, 1],
        target_names=["SUPPORTED", "HALLUCINATED"],
        zero_division=0
    )
)

print("\nSTAGE 9E COMPLETED SUCCESSFULLY!")


COLLECTING TEST PREDICTIONS
Test batch 100/675
Test batch 200/675
Test batch 300/675
Test batch 400/675
